In [1]:

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')


In [2]:
import joblib

X_train_scaled = joblib.load("../test_train/X_train_scaled.pkl")
X_test_scaled = joblib.load("../test_train/X_test_scaled.pkl")
y_train = joblib.load("../test_train/y_train.pkl")
y_test = joblib.load("../test_train/y_test.pkl")

X_train_balanced = joblib.load("../test_train/X_train_balanced.pkl")
y_train_balanced = joblib.load("../test_train/y_train_balanced.pkl")

feature_columns = joblib.load("../test_train/feature_columns.pkl")

print("Loaded all train/test files from 'test_train'")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
print("X_train_balanced shape:", X_train_balanced.shape)
print("y_train_balanced shape:", y_train_balanced.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Loaded all train/test files from 'test_train'
X_train_scaled shape: (356058, 82)
X_test_scaled shape: (89015, 82)
X_train_balanced shape: (1106808, 82)
y_train_balanced shape: (1106808,)
y_train shape: (356058,)
y_test shape: (89015,)


## TRAINING UNBALANCED MODELS

In [3]:

 
# ============================================================================
#  TRAIN  MODELS 
# ============================================================================
print("="*80)
print(" TRAINING MODELS")
print("="*80)
 
models = {}
cv_scores = {}
 
# ----------------------------------------------------------------------------
# Model 1: Logistic Regression (Baseline)
# ----------------------------------------------------------------------------
 
lr_model = LogisticRegression(
    solver='saga',
    max_iter=500,  
    random_state=42,
    n_jobs=-1,
    verbose=0
)
 
lr_model.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr_model
 
# Quick 3-fold CV on subset for speed
cv_score = cross_val_score(
    lr_model, X_train_scaled, y_train, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['Logistic Regression'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")
 
# ----------------------------------------------------------------------------
# Model 2: Decision Tree
# ----------------------------------------------------------------------------
print("\n Training Decision Tree...")
 
dt_model = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=100,
    min_samples_leaf=50,
    random_state=42
)
 
dt_model.fit(X_train_scaled, y_train)
models['Decision Tree'] = dt_model
 
cv_score = cross_val_score(
    dt_model, X_train_scaled, y_train, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['Decision Tree'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")

 
# ----------------------------------------------------------------------------
# Model 3: Random Forest (Default)
# ----------------------------------------------------------------------------
print("\n Training Random Forest (default)...")
 
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=100,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
 
rf_model.fit(X_train_scaled, y_train)
models['Random Forest'] = rf_model
 
cv_score = cross_val_score(
    rf_model, X_train_scaled, y_train, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['Random Forest'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")
 
 


 


 

 TRAINING MODELS
  ✓ Training complete
  CV F1-Score (macro): 0.2649 ± 0.0008

 Training Decision Tree...
  ✓ Training complete
  CV F1-Score (macro): 0.3801 ± 0.0027

 Training Random Forest (default)...
  ✓ Training complete
  CV F1-Score (macro): 0.3155 ± 0.0014


In [5]:
 
# ----------------------------------------------------------------------------
# Model 4: XGBoost (Default)
# ----------------------------------------------------------------------------
print("\n Training XGBoost (default)...")
# Re-label target for XGBoost
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

print("Original classes:", sorted(y_train.unique()))
print("XGBoost classes:", sorted(y_train_xgb.unique()))

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(X_train_scaled, y_train_xgb)
models['XGBoost'] = xgb_model
 
cv_score = cross_val_score(
    xgb_model, X_train_scaled, y_train_xgb, 
    cv=3, scoring='f1_macro', n_jobs=-1
)
cv_scores['XGBoost'] = cv_score
 
print(f"  ✓ Training complete")
print(f"  CV F1-Score (macro): {cv_score.mean():.4f} ± {cv_score.std():.4f}")


 Training XGBoost (default)...
Original classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
XGBoost classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  ✓ Training complete
  CV F1-Score (macro): 0.4245 ± 0.0024


## EVALUATION

In [7]:

models ={
    'Logistic Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}
results = []
all_predictions = {}
for model_name, model in models.items():
    print(f"\nEvaluating: {model_name}...")

    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)
    
    # Adjust labels for XGBoost (it was trained on 0-indexed labels)
    if "XGBoost" in model_name:
        y_pred = y_pred + 1
    
    # Store predictions
    all_predictions[model_name] = {
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)

    try:
        roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
    except:
        roc_auc = 0.0
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'F1-Score (macro)': f1_macro,
        'F1-Score (weighted)': f1_weighted,
        'Precision (macro)': precision_macro,
        'Recall (macro)': recall_macro,
        'ROC-AUC': roc_auc
    })
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score (macro): {f1_macro:.4f}")
    print(f"  F1-Score (weighted): {f1_weighted:.4f}")


Evaluating: Logistic Regression...
  Accuracy: 0.7819
  F1-Score (macro): 0.2660
  F1-Score (weighted): 0.7145

Evaluating: Decision Tree...
  Accuracy: 0.8201
  F1-Score (macro): 0.3878
  F1-Score (weighted): 0.7966

Evaluating: Random Forest...
  Accuracy: 0.8082
  F1-Score (macro): 0.3205
  F1-Score (weighted): 0.7628

Evaluating: XGBoost...
  Accuracy: 0.8346
  F1-Score (macro): 0.4259
  F1-Score (weighted): 0.8113


In [12]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
 
# Helper function to get correct y_test for each model
def get_labels(model_name, y_test):
    """Handle XGBoost label offset (0-3 vs 1-4)"""
    is_xgboost = 'XGBoost' in model_name
    return y_test - 1 if is_xgboost else y_test, [0,1,2,3] if is_xgboost else [1,2,3,4]
 
# ============================================================================
# CONFUSION MATRICES (Top 3 Models)
# ============================================================================
print("Creating confusion matrices...")
 
top_models = ['Random Forest', 'Decision Tree', 'XGBoost']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
for idx, model_name in enumerate(top_models):
    model = models[model_name]
    y_true, labels = get_labels(model_name, y_test)
    
    # Predict and create confusion matrix
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', 
                xticklabels=['Sev 1', 'Sev 2', 'Sev 3', 'Sev 4'],
                yticklabels=['Sev 1', 'Sev 2', 'Sev 3', 'Sev 4'],
                ax=axes[idx], cbar_kws={'label': 'Rate'})
    
    axes[idx].set_title(f'{model_name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('Actual', fontsize=10)
 
plt.suptitle('Confusion Matrices: Top 3 Models (Unbalanced)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('unbalanced_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✓ Saved: unbalanced_confusion_matrices.png\n")
plt.close()
 
# ============================================================================
# ROC CURVES (Best Model: XGBoost)
# ============================================================================
print("Creating ROC curves for best model (XGBoost)...")
 
model = models['XGBoost']
y_true, labels = get_labels('XGBoost', y_test)
 
# Get prediction probabilities
y_proba = model.predict_proba(X_test_scaled)
 
# Create 2x2 subplot for each severity class
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
 
severity_names = ['Severity 1', 'Severity 2', 'Severity 3', 'Severity 4']
 
for idx, (severity_label, severity_name) in enumerate(zip(labels, severity_names)):
    ax = axes[idx]
    
    # One-vs-Rest: binary classification for this severity
    y_binary = (y_true == severity_label).astype(int)
    y_score = y_proba[:, idx]
    
    # Calculate ROC
    fpr, tpr, _ = roc_curve(y_binary, y_score)
    auc = roc_auc_score(y_binary, y_score)
    
    # Plot
    ax.plot(fpr, tpr, linewidth=2.5, color='#2ecc71', label=f'AUC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5, label='Random')
    
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
    ax.set_title(f'{severity_name} (One-vs-Rest)', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
 
plt.suptitle('ROC Curves: XGBoost- Best Model(Unbalanced)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('unbalanced_roc_curves_best_model.png', dpi=300, bbox_inches='tight')
print("✓ Saved: unbalanced_roc_curves_best_model.png\n")
plt.close()
 
# ============================================================================
# PER-CLASS PERFORMANCE BREAKDOWN
# ============================================================================
print("Creating per-class performance analysis...")
 
from sklearn.metrics import classification_report
 
# Get detailed metrics for best model
model = models['XGBoost']
y_true, labels = get_labels('XGBoost', y_test)
y_pred = model.predict(X_test_scaled)
 
# Classification report
report = classification_report(y_true, y_pred, labels=labels, 
                               target_names=severity_names, 
                               output_dict=True)
 
# Extract metrics
metrics_df = pd.DataFrame({
    'Severity': severity_names,
    'Precision': [report[name]['precision'] for name in severity_names],
    'Recall': [report[name]['recall'] for name in severity_names],
    'F1-Score': [report[name]['f1-score'] for name in severity_names],
    'Support': [report[name]['support'] for name in severity_names]
})
 
print("\nPer-Class Performance (XGBoost):")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)
 
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(severity_names))
width = 0.25
 
bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision', color='#3498db')
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall', color='#2ecc71')
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#f39c12')
 
ax.set_xlabel('Severity Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance: XGBoost', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(severity_names)
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
 
# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)
 
plt.tight_layout()
plt.savefig('unbalanced_per_class_performance.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: unbalanced_per_class_performance.png")
plt.close()
 
# Save metrics to CSV
metrics_df.to_csv('unbalanced_per_class_metrics.csv', index=False)
print("✓ Saved: unbalanced_per_class_metrics.csv")
 

 

Creating confusion matrices...
✓ Saved: unbalanced_confusion_matrices.png

Creating ROC curves for best model (XGBoost)...
✓ Saved: unbalanced_roc_curves_best_model.png

Creating per-class performance analysis...

Per-Class Performance (XGBoost):
  Severity  Precision   Recall  F1-Score  Support
Severity 1   0.595745 0.066272  0.119276    845.0
Severity 2   0.851691 0.957832  0.901649  69176.0
Severity 3   0.721331 0.469040  0.568450  16683.0
Severity 4   0.536232 0.064042  0.114418   2311.0

✓ Saved: unbalanced_per_class_performance.png
✓ Saved: unbalanced_per_class_metrics.csv
